In [1]:
import numpy as np 
import pandas as pd 
from collections import Counter
data_cart = [
    {'Age':'<=30', 'Income':'High',   'Student':'No',  'Credit':'Fair',      'BuysComputer':'No'},
    {'Age':'<=30', 'Income':'High',   'Student':'No',  'Credit':'Excellent', 'BuysComputer':'No'},
    {'Age':'31-40','Income':'High',   'Student':'No',  'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'No',  'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Low',    'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Low',    'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'No'},
    {'Age':'31-40','Income':'Low',    'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'<=30', 'Income':'Medium', 'Student':'No',  'Credit':'Fair',      'BuysComputer':'No'},
    {'Age':'<=30', 'Income':'Low',    'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'<=30', 'Income':'Medium', 'Student':'Yes', 'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'31-40','Income':'Medium', 'Student':'No',  'Credit':'Excellent', 'BuysComputer':'Yes'},
    {'Age':'31-40','Income':'High',   'Student':'Yes', 'Credit':'Fair',      'BuysComputer':'Yes'},
    {'Age':'>40',  'Income':'Medium', 'Student':'No',  'Credit':'Excellent', 'BuysComputer':'No'},
]

In [ ]:
def gini(df, attr):
    subset = df[attr]
    counts = Counter(subset)
    total = sum(counts.values())
    g = 1
    for c, v in counts.items():
        g -= (v/total)**2
    
    return g

In [8]:
df = pd.DataFrame(data_cart)
gini(df, 'BuysComputer')

0.4591836734693877

In [12]:
def gini_subset(df, attr, value, target):
    subset = df[df[attr] == value]
    column = subset[target]
    counts = Counter(column)
    total = sum(counts.values())
    g = 1
    for c, v in counts.items():
        g -= (v/total)**2
    
    return g

In [13]:
gini_subset(df, 'Age','<=30','BuysComputer')

0.48

In [16]:
def gini_split(df, attr, target):
    n = len(df)
    weighted = 0
    for val in df[attr].unique():
        subset = df[df[attr] == val]
        weight = len(subset) / n
        weighted += weight * gini_subset(df, attr, val, target)
    return weighted


In [17]:
gini_split(df, 'Age', 'BuysComputer')

0.34285714285714286

In [19]:
def best_split(df, target):
    best_attr = None 
    best_gini = float("inf")

    for attr in df.columns:
        if attr == target:
            continue
        else:
            best_gini_curr = gini_split(df, attr, target)
            if best_gini_curr <= best_gini:
                best_gini = best_gini_curr
                best_attr = attr 
    
    return best_gini, best_attr

In [20]:
best_split(df, "BuysComputer")

(0.34285714285714286, 'Age')

In [21]:

def build_cart_tree(df, target, attributes=None):
    # Consider only the available attributes
    if attributes is None:
        attributes = [col for col in df.columns if col != target]

    # Base case 1: all examples belong to the same class
    labels = df[target].value_counts()

    if len(labels) == 1:
        return labels.index[0]

    # Base case 2: no attributes remain
    if not attributes:
        return labels.index[0]  # Majority class

    # Find the best attribute using weighted Gini impurity
    best_attr, best_gini = best_split(
        df[attributes + [target]], target
    )

    # Base case 3: no valid split
    if best_attr is None:
        return labels.index[0]

    # Create a decision node
    tree = {best_attr: {}}

    # Remove the selected attribute for child nodes
    remaining_attributes = [
        attr for attr in attributes if attr != best_attr
    ]

    # Recursively build a subtree for every attribute value
    for value in df[best_attr].unique():
        subset = df[df[best_attr] == value]

        if subset.empty:
            tree[best_attr][value] = labels.index[0]
        else:
            tree[best_attr][value] = build_cart_tree(
                subset,
                target,
                remaining_attributes
            )

    return tree


In [22]:
build_cart_tree(df, 'BuysComputer')

KeyError: 0.34285714285714286